# Лекция 11: Компьютерное зрение
## Адаптеры в диффузионных моделях & SOTA модели. Часть 3

**Преподаватель:** Шевченко Елена, Т-Банк

**Аудитория:** ФКН ВШЭ, МФТИ

---

## Содержание

1. ~~Адаптеры~~
   1. ~~Зачем нужны, типы адаптеров~~
   2. ~~ControlNet~~
   3. ~~IP-Adapter~~
   4. ~~InstantStyle: инъекция по блокам~~
   5. ~~Что ещё бывает~~
   6. ~~От адаптеров к in-context conditioning~~
2. ~~Как устроена современная text-to-image модель~~
   0. ~~Общая схема~~
   1. ~~Flow Matching~~
   2. ~~Картинка — это последовательность токенов~~
   3. ~~Скалярные условия — через adaLN~~
   4. ~~RoPE — позиции токенов~~
   5. ~~Текст — через joint attention~~
   6. ~~DoubleStream и SingleStream блоки~~
   7. ~~Референсы — это тоже токены~~
   8. ~~Меньше шагов~~
   9. ~~Итог~~
3. **Современные модели: FLUX и Qwen-Image** (этот ноутбук)
   1. FLUX.1 — эталон
   2. FLUX.1 Kontext — редактирование через референсы
   3. FLUX.2 — больше, проще, с VLM
   4. FLUX 3 Klein — маленькая и быстрая
   5. Qwen-Image — другая ветка
   6. Закрытые модели
   7. Итог
4. Дистилляция

---

В 11-2 разобрали детали, из которых собрана любая современная text-to-image модель. Здесь — как из них собраны конкретные модели. Для каждой — только то, что в ней нового.

```
BFL:    FLUX.1 (2024-08) → FLUX.1 Kontext (2025) → FLUX.2 (2025-11) → FLUX 3 Klein (2026)
Alibaba: Qwen-Image (2025-08) → Qwen-Image-Edit → Qwen-Image-2.0 (2026-02) → Qwen-Image-2.1 (2026-09)
```

## 1. FLUX.1 — эталон

![flux1](assets/flux1.png)

Black Forest Labs (BFL), август 2024. Это почти дословно сумма принципов из 11-2, поэтому FLUX.1 удобно держать в голове как точку отсчёта.

| часть | FLUX.1 |
|---|---|
| размер | 12B, ширина 3072 (24 головы × 128) |
| блоки | 19 DoubleStream + 38 SingleStream |
| текст | T5-XXL — токены; CLIP-L — pooled-вектор в adaLN |
| VAE | ×8, 16 каналов → 4096 токенов на 1024² |
| позиции | RoPE по трём осям $(i, h, w)$; в text-to-image $i = 0$ |
| MLP | GELU |

### Вектор условия

Для adaLN он складывается из трёх частей:

$$c = \text{MLP}_t(t) + \text{MLP}_g(g) + \text{MLP}_y(\text{CLIP}_{\text{pooled}})$$

$g$ — масштаб guidance. FLUX.1 [dev] обучен выдавать результат CFG за один проход: $g$ подаётся на вход через $\text{MLP}_g$, поэтому шаг вдвое дешевле (guidance distillation, подробно — в 11-4).

### Варианты
- [pro] — только API;
- [dev] — открытые веса, некоммерческая лицензия, 20–50 шагов;
- [schnell] — Apache 2.0, 1–4 шага (adversarial distillation, в 11-4).

## 2. FLUX.1 Kontext — редактирование через референсы

![flux-kontext](assets/flux-kontext.png)

2025 год. Та же сеть, что FLUX.1, без новых модулей. Это ровно принцип «референсы — это тоже токены» в чистом виде:

```
FLUX.1:   [ текст | генерируемая картинка ]
Kontext:  [ текст | генерируемая картинка | референс ]
```

- Референс кодируется тем же VAE и дописывается в последовательность без шума.
- Номер изображения — та самая первая ось RoPE, которая во FLUX.1 всегда была 0: генерируемая картинка $(0, h, w)$, референс $(1, h, w)$.
- Лосс только по токенам генерируемой картинки. Без референса модель остаётся обычной text-to-image.

Авторы пробовали склеивать референс с картинкой по каналам — получилось хуже, чем по длине последовательности.

### Что умеет

Правка по инструкции («убери машину»), замена текста на картинке, тот же персонаж в новой сцене, перенос стиля, несколько правок подряд.

### Зачем это нужно

Раньше редактировали через inversion: зашумить исходник и сгенерировать заново с другим промптом. Меняется вся картинка. Kontext видит исходник на каждом шаге и копирует всё, что не нужно менять.

## 3. FLUX.2 — больше, проще, с VLM

![flux2](assets/flux2.png)

Ноябрь 2025. Генерация и редактирование — в одной модели. Что изменилось относительно FLUX.1:

| | FLUX.1 [dev] | FLUX.2 [dev] |
|---|---|---|
| размер | 12B | 32B |
| блоки | 19 Double + 38 Single | 8 Double + 48 Single |
| текст | T5 + CLIP | Mistral Small 3.2 (24B), три слоя |
| VAE | 16 каналов | 32 канала |
| MLP | GELU | SwiGLU |
| modulation | своя в каждом блоке | три на всю сеть |
| референсы | отдельная модель Kontext | встроены, до 10 |
| RoPE | 3 оси $(i, h, w)$, у текста позиций нет | 4 оси $(t, h, w, l)$: текст — по оси $l$, референс $k$ — $t = 10(k+1)$ |

### Текстовый энкодер

Вместо T5 + CLIP — одна VLM. Берут скрытые состояния слоёв 10, 20 и 30, склеивают по каналам и проецируют Linear в ширину трансформера. Pooled-вектора больше нет: в adaLN идут только $t$ и guidance.

### SwiGLU вместо GELU

Тот же MLP, что в современных LLM:

$$\text{MLP}(x) = W_2\, \text{GELU}(W_1 x) \quad\to\quad \text{SwiGLU}(x) = W_2\, \big(\text{SiLU}(W_g x) \odot W_1 x\big)$$

где

$$\text{GELU}(x) = x \cdot \Phi(x), \qquad \text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}}$$

$\Phi$ — функция распределения $\mathcal{N}(0, 1)$, $\sigma$ — сигмоида. Обе функции — гладкие версии ReLU, по форме почти одинаковые. Разница не в активации, а в устройстве:
- в GELU-MLP вход проходит одну проекцию $W_1$ и активацию;
- в SwiGLU две проекции: $W_1 x$ — значения, $\text{SiLU}(W_g x)$ — «вентиль», который поэлементно решает, сколько от каждого значения пропустить.

В LLaMA скрытую размерность SwiGLU берут в $2/3$ от GELU-MLP, чтобы число параметров не выросло: матриц три вместо двух. Во FLUX.2 так не делают: скрытая размерность $3d$ (во FLUX.1 было $4d$), поэтому MLP даже немного больше — $9d^2$ параметров против $8d^2$.

### Общая modulation

Вектор условия $c$ один и тот же для всех блоков, но во FLUX.1 каждый блок $b$ делал из него свой Linear:

$$\text{FLUX.1:}\ (\gamma_b, \beta_b, \alpha_b) = W_b\, c \qquad\to\qquad \text{FLUX.2:}\ (\gamma, \beta, \alpha) = W c \ \text{ для всех блоков типа}$$

Сколько это стоит во FLUX.1 (ширина 3072; DoubleStream — 6 векторов на каждый из двух потоков, SingleStream — 3):

$$19 \cdot 2 \cdot (3072 \cdot 6 \cdot 3072) + 38 \cdot (3072 \cdot 3 \cdot 3072) \approx 2.15\text{B} + 1.08\text{B} \approx 3.2\text{B}$$

То есть около четверти из 12B параметров уходило только на modulation. Во FLUX.2 modulation-модулей три на всю сеть: один для токенов картинки и один для токенов текста во всех DoubleStream-блоках, один для всех SingleStream. Заодно убраны bias во всех линейных слоях.

### Референсы

Та же схема, что в Kontext, но уже в базовой модели и до 10 изображений. Номер референса $k$ кодируется смещением по первой оси RoPE: $t = 10(k+1)$.

### Версии

[pro], [flex] — API; [dev] — открытые веса, некоммерческая лицензия; **[klein] 4B и 9B** — маленькие открытые версии (4B — Apache 2.0, 9B — некоммерческая лицензия), у каждой два варианта: distilled (4 шага — для генерации) и base (без дистилляции, ~50 шагов — для дообучения и LoRA). Дообучать distilled-модель плохо: дистилляция «запекает» в веса короткую траекторию и CFG, и дообучение это ломает.

## 4. FLUX 3 Klein — маленькая и быстрая

![flux3-o](assets/flux3-overview.png)

FLUX 3 (анонс — июль 2026) — уже не только картинки. Это одна модель, обученная вместе на изображениях, видео и аудио; её же расширяют до предсказания действий робота. Отсюда семейство:
- **FLUX 3 Video** — видео до 20 секунд со звуком: речь с синхронизацией губ, эффекты, фон генерируются вместе с кадрами; первая видеомодель BFL;
- **FLUX 3 Image** — картинки, вышел в октябре 2026;
- **FLUX 3 Action** — предсказание действий для роботов.

Основная часть обучения ушла на видео (по словам BFL, больше 95% вычислений). Архитектура мультимодальной модели не раскрыта; ниже — только то, что видно в коде открытой модели для картинок.

У FLUX 3 Image открытые веса есть только у маленькой версии — Klein, в двух вариантах: base и distilled — та же пара, что у FLUX.2 [klein]. Технического отчёта нет, архитектура известна из кода. Лицензия — FLUX Kommunity License v1.0: коммерческое использование разрешено при годовой выручке до USD 5M.

![p_flux3](assets/p_flux3.png)

| | FLUX.2 [dev] | FLUX 3 Klein |
|---|---|---|
| размер | 32B | 5.6B |
| текст | Mistral-3, 3 слоя | Qwen3-VL-4B, 8 из 36 слоёв |
| VAE | ×8, 32 канала | тот же |
| структура | 8 Double + 48 Single | 3 потока × 5 блоков, потом 28 общих |
| блоки | Single — параллельные | все — параллельные |
| timestep | один на все токены | свой у каждого типа токенов |
| шаги | 50 | base 50 с CFG; distilled 16 без CFG |

### Текстовый энкодер

Берут скрытые состояния каждого четвёртого слоя Qwen3-VL-4B, склеивают по каналам и проецируют в ширину трансформера:

$$e = W\,[\,h_4;\ h_8;\ \dots;\ h_{32}\,], \qquad 8 \cdot 2560 = 20480 \ \to\ 3072$$

### Две стадии

Сначала текст, картинка и референсы обрабатываются каждый в своём потоке, по 5 блоков, и attention только внутри потока. Потом 28 общих блоков, где все токены видят всех. Во FLUX.2 обмен шёл с первого блока. В конце через выходной слой проходят только токены картинки.

### Свой timestep у каждого типа токенов

Во FLUX.1 и FLUX.2 одно $t$ модулирует все токены. Во FLUX 3 токены генерируемой картинки получают текущее $t$, а текст и референсы — всегда $t = 0$, то есть «без шума»:

$$c_{\text{img}} = \text{MLP}(t), \qquad c_{\text{txt}} = c_{\text{ref}} = \text{MLP}(0)$$
Так модель различает референс и результат, даже когда они почти совпадают. Это для base-версии; в distilled к каждому $c$ прибавляется ещё эмбеддинг целевого времени $s$. Scale / shift / gate у текста, картинки и референсов считают свои головы.

### RoPE по четырём осям

Оси $(t, h, w, l)$:

```
текст,     токен i:         (0,   0,     0,     i)
картинка,  позиция (h, w):  (0,   h/2,   w/2,   0)
референс k, позиция (h, w): (0,   h/2,   w/2,  −(k+1))
```

Отличия от FLUX.2: номер референса переехал с оси $t$ (там $10(k+1)$) на ось $l$, а координаты $h, w$ умножаются на 0.5. Референс и результат совпадают по $(h, w)$ и различаются только осью $l$, поэтому модель легко находит, какой участок референса соответствует какому участку результата.

Редактирование по рамкам («перерисуй вот этот прямоугольник») устроено не через RoPE: рамки пользователь задаёт в долях 0–1, prompt upsampler переводит их на сетку 0–1000 и вписывает в текст промпта. Это подсказка модели, а не жёсткая маска.

### Flow map в distilled-версии

Сеть получает не только текущее время $t$, но и время $s$, в которое прыгает шаг, и предсказывает среднюю скорость на всём отрезке:

$$x_s = x_t - (t - s)\, u_\theta(x_t, s, t), \qquad s < t$$

Обычная модель знает только мгновенную скорость в точке, поэтому шаги должны быть маленькими. Flow map знает, куда приведёт большой шаг, — отсюда 16 шагов без CFG. Подробно — в 11-4.

### Self-Flow

BFL сообщает, что FLUX 3 обучали с Self-Flow — меняется только обучение, не архитектура. Идея: модель сама себе учитель. Случайной доле токенов (до половины) дают уровень шума $s$, остальным — $t$; оба уровня берут из одного распределения. Студент видит такой неоднородно зашумлённый вход и учится предсказывать признаки своей EMA-копии с более глубокого слоя. Учитель видит весь вход с уровнем $\min(t, s)$, то есть зашумлённый слабее. Лосс:

$$\mathcal{L} = \mathcal{L}_{\text{FM}} + \lambda\, \mathcal{L}_{\text{rep}}, \qquad \mathcal{L}_{\text{rep}} = 1 - \cos\big(P(f^{(l)}_{\text{студент}}),\ f^{(L)}_{\text{учитель}}\big), \quad l < L$$

где $f^{(l)}$ — признаки слоя $l$, $P$ — небольшая проекция, учитель — EMA-копия модели.

Раньше похожий трюк (REPA) требовал внешнего энкодера вроде DINOv2. Self-Flow обходится без него; по данным авторов, сходимость примерно в 2.8 раза быстрее, чем у REPA.

### Железо

Пик памяти в bf16 на 1024² без prompt upsampler — около 31 GB на B200 по карточке модели и около 20.7 GiB на H100 по блогу HF. Upsampler — отдельная 9B-модель, ещё около 28 GB. На Colab T4 (16 GB) Klein не помещается.

## 5. Qwen-Image — другая ветка

![qwen](assets/qwen.jpg)

Alibaba, команда Qwen. Те же принципы, но другие решения.

### Qwen-Image v1 (август 2025)

| часть | Qwen-Image v1 |
|---|---|
| размер | 20B |
| блоки | 60 DoubleStream, SingleStream нет |
| текст | Qwen2.5-VL-7B, заморожен, последний слой |
| VAE | ×8, 16 каналов; декодер дообучен на картинках с текстом |
| позиции | MSRoPE |
| лицензия | Apache 2.0 |

Главная сила — рендеринг текста на картинке, в том числе китайского. На это работают VLM-энкодер и декодер VAE, дообученный на документах, слайдах и постерах.

#### MSRoPE

Позиции картинки отсчитываются от её центра. Текстовые токены получают одинаковые координаты по обеим осям — как будто текст записан по диагонали картинки. Для самого текста это обычный 1D-RoPE, а картинка и текст не путаются позициями.

### Qwen-Image-Edit: референс идёт двумя путями

```
                     ┌─→ Qwen2.5-VL ─→ поток текста (вместе с инструкцией)
исходная картинка ───┤
                     └─→ VAE ────────→ поток картинки, рядом с генерируемой
```

- Путь через VAE — как в Kontext: токены референса в общей последовательности плюс ось RoPE для номера изображения. Он отвечает за внешний вид: цвета, текстуры, детали.
- Путь через VLM — новое. Он отвечает за смысл: что изображено и что именно менять.

Оба пути встречаются в обычном joint attention, отдельного cross-attention нет.

### Qwen-Image-2.0 (февраль 2026, только API)

Одна модель на генерацию и редактирование. Техотчёт есть, размер модели в нём не указан.

| | Qwen-Image v1 | Qwen-Image-2.0 |
|---|---|---|
| текст | Qwen2.5-VL-7B | Qwen3-VL, заморожен |
| бэкбон | два потока | один поток |
| MLP | GELU | SwiGLU |
| modulation | scale, shift и gate | без shift |
| VAE | ×8, 16 каналов | ×16, 64 канала |
| шаги | десятки | 4 после дистилляции |

- **Modulation без сдвига** — остаётся только масштаб:

$$h = (1 + \text{scale}) \odot \text{LN}(x) + \text{shift} \quad\to\quad h = (1 + \text{scale}) \odot \text{LN}(x)$$

- **VAE ×16.** Вчетверо меньше токенов на ту же картинку. Чтобы не потерять детали — 64 канала и дополнительный лосс, который выравнивает латент по смыслу. Adversarial loss убран ради стабильности.
- **Обучение** в три этапа: от 256–512 к 2048 пикселям, затем SFT. После — RL (вариант GRPO для диффузии) с наградами за эстетику, соответствие тексту и следование инструкции.
- **Дистилляция** DMD: учитель делает 40 шагов, студент — 4. Подробно — в 11-4.
- **Prompt enhancer** — отдельная LLM, которая переписывает короткий запрос пользователя в подробный.

### Qwen-Image-2.1 (сентябрь 2026)

Открытые веса вернулись: 7B, генерация и редактирование в одной модели, есть в diffusers. Лицензия исследовательская — для коммерческого использования нужна отдельная.

## 6. Закрытые модели

Верх рейтинга [Arena text-to-image](https://arena.ai/leaderboard/image/text-to-image) — закрытые модели: GPT-Image (OpenAI), Nano Banana (Google), Seedream (ByteDance), Grok Imagine, MAI-Image (Microsoft), Reve, Qwen-Image-3.0-Pro. Архитектуры не раскрыты, поэтому разбирать их нечего. Рейтинг строится по попарным голосам людей — та же модель Bradley-Terry, что в лекции 9-1.

## 7. Итог

| | FLUX.1 | FLUX.2 | Qwen-Image-2.0 | FLUX 3 Klein |
|---|---|---|---|---|
| размер | 12B | 32B | не раскрыт | 5.6B |
| текст | T5 + CLIP | Mistral-3 | Qwen3-VL | Qwen3-VL-4B |
| VAE | ×8, 16 | ×8, 32 | ×16, 64 | ×8, 32 |
| референсы | через Kontext | встроены | VAE + VLM | свой поток, $t = 0$ |
| шаги | 20–50 | 50 | 4 | 16 |
| веса | [dev] — некоммерч., [schnell] — Apache 2.0 | [dev], [klein] 9B — некоммерч.; [klein] 4B — Apache 2.0 | API | FLUX Kommunity License |

Куда всё движется:
- **текст** — VLM вместо T5 + CLIP, и не один слой, а несколько;
- **блоки** — у FLUX всё больше общих, всё меньше раздельных (Qwen-Image идёт иначе — все блоки DoubleStream);
- **VAE** — больше каналов, меньше токенов;
- **редактирование** — не отдельной моделью и не адаптером, а референсами в той же последовательности;
- **шаги** — дистилляция до 4–16 шагов без CFG (11-4).

## Источники

- FLUX.1: код инференса — https://github.com/black-forest-labs/flux
- FLUX.1 Kontext: https://arxiv.org/abs/2506.15742
- FLUX.2, FLUX 3 Klein: код diffusers (`transformer_flux2.py`, `transformer_flux3.py`), https://huggingface.co/black-forest-labs/flux-3-image-klein, https://github.com/huggingface/blog/blob/main/flux-3.md
- FLUX 3 (анонс): https://www.globenewswire.com/news-release/2026/07/23/3332364/0/en/black-forest-labs-unveils-flux-3-a-new-multimodal-frontier-model-for-visual-intelligence.html, https://docs.bfl.ml/release-notes
- Self-Flow: https://arxiv.org/abs/2603.06507; FLUX 3 построен на Self-Flow, >95% вычислений — на видео: https://bfl.ai/blog/flux-3, https://bfl.ai/blog/flux-3-mimic
- Qwen-Image: https://arxiv.org/abs/2508.02324
- Qwen-Image-2.0: https://arxiv.org/abs/2605.10730
- Qwen-Image-2.1: https://huggingface.co/Qwen/Qwen-Image-2.1
- Arena: https://arena.ai/leaderboard/image/text-to-image